# Logistic Regression from Scratch

This notebook trains a simple event-level classifier for the triangular statistical arbitrage research system.

The target is the event label defined during labeling:

\[
y_i = 1 \quad \text{if the residual reverts before stop-loss}
\]

The model estimates:

\[
P(y_i = 1 \mid x_i)
\]

where \(x_i\) is the feature vector available at the candidate event date.

## Research constraint

The event matrix is sorted by time and split without shuffling. Shuffling would mix regimes and make validation less realistic for a time-series trading problem.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.logistic_model import (
    LogisticConfig,
    calibration_table,
    train_event_logistic_model,
)

In [ ]:
DATA_DIR = Path("../data/processed") if Path("../data/processed").exists() else Path("data/processed")
FIGURE_DIR = Path("../figures") if Path("../figures").exists() else Path("figures")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

feature_path = DATA_DIR / "event_feature_matrix.csv"
feature_matrix = pd.read_csv(feature_path)
feature_matrix["event_date"] = pd.to_datetime(feature_matrix["event_date"])
feature_matrix.head()

## Model fit

The implementation uses the sigmoid link, binary cross-entropy, batch gradient descent, and L2 regularization. The model is intentionally simple so that probability estimates can be inspected before moving to more complex classifiers.

In [ ]:
config = LogisticConfig(
    learning_rate=0.05,
    max_iter=800,
    l2_penalty=0.25,
    tolerance=1e-10,
    standardize=True,
    threshold=0.50,
)

outputs = train_event_logistic_model(feature_matrix, config=config)

coefficients = outputs["model_coefficients"]
loss_history = outputs["training_loss"]
predictions = outputs["predicted_reversion_probabilities"]
metrics = outputs["validation_metrics"]

coefficients.head(10)

In [ ]:
loss_history.tail()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history["iteration"], loss_history["loss"], linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Logistic regression training loss")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "logistic_loss_curve.png", dpi=160)
plt.show()

## Validation metrics

The validation and test metrics should be interpreted as diagnostics. Small event samples can produce unstable precision, recall, and AUC values. A larger walk-forward evaluation is needed before using model probabilities in backtest filtering.

In [ ]:
metrics

In [ ]:
calibration = calibration_table(
    predictions["label"],
    predictions["predicted_reversion_probability"],
    n_bins=5,
)
calibration

In [ ]:
coefficients.to_csv(DATA_DIR / "logistic_model_coefficients.csv", index=False)
loss_history.to_csv(DATA_DIR / "logistic_training_loss.csv", index=False)
predictions.to_csv(DATA_DIR / "predicted_reversion_probabilities.csv", index=False)
metrics.to_csv(DATA_DIR / "logistic_validation_metrics.csv", index=False)
calibration.to_csv(DATA_DIR / "logistic_probability_calibration.csv", index=False)

## Interpretation boundary

The probability output is not a trading recommendation. It is a research signal estimating the historical likelihood that an event with similar features reverted before its stop-loss under the tested labeling rules.